In [2]:
import pandas as pd

file_path = r"C:\Users\ADMIN\Documents\mini_project_guvi\project_ecotype\classification.csv" 
data = pd.read_csv(file_path)
data.head()

,Elevation,Horizontal_Distance_To_Roadways,Horizontal_Distance_To_Fire_Points,Hydrology_Distance,Hydrology_Ratio,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Aspect_sin,Elevation_Slope,Hillshade_Mean,...,Hillshade_3pm,Hillshade_9am,Slope,Soil_Type_12.0,Soil_Type_10.0,Soil_Type_29.0,Soil_Type_3.0,Soil_Type_23.0,Soil_Type_30.0,Cover_Type
0,2596.0,510.0,6279.0,1.173173,-0.727312,0.258124,-0.915049,-0.013352,-4560.885708,49.468393,...,148.0,0.014542,-1.756890,False,False,True,False,False,False,0
1,2590.0,390.0,6225.0,1.241814,-1.188005,0.024593,-1.217221,-0.012005,-5398.801061,50.524699,...,151.0,-0.042148,-2.084479,False,False,True,False,False,False,0
2,2804.0,3180.0,6121.0,1.236781,0.712770,0.305943,0.930837,0.004059,-1022.254664,45.560944,...,135.0,0.832835,-0.364570,True,False,False,False,False,False,4
3,2785.0,3090.0,6211.0,2.361370,1.849682,0.179560,2.181810,0.006401,2880.056772,41.322305,...,122.0,1.116918,1.034132,False,False,False,False,False,True,4
4,2595.0,391.0,6172.0,0.642165,-1.407147,-0.317796,-0.959961,-0.015077,-5409.223457,50.165999,...,150.0,-0.042148,-2.084479,False,False,True,False,False,False,0


# checking target class is balanced or imbalanced

In [3]:
data['Cover_Type'].value_counts()

Cover_Type
4    103071
6     31110
0      3069
3      2160
5      2160
2      2160
1      2160
Name: count, dtype: int64

## **Handling class imbalance**
# SMOTE creates synthetic samples, not duplicates.
# Counter to check the class distribution before and after applying SMOTE to confirm whether the dataset is balanced.

In [4]:
from imblearn.over_sampling import SMOTE
from collections import Counter

In [5]:
from sklearn.model_selection import train_test_split

# Example: Define X and y
X = data.drop("Cover_Type", axis=1)   # Features
y = data["Cover_Type"]                # Target variable

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Before SMOTE:", Counter(y_train))

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:", Counter(y_train_smote))

Before SMOTE: Counter({4: 82457, 6: 24888, 0: 2455, 5: 1728, 1: 1728, 3: 1728, 2: 1728})
After SMOTE: Counter({4: 82457, 5: 82457, 1: 82457, 6: 82457, 3: 82457, 0: 82457, 2: 82457})


# Model Build-"I built five classification models and compared them using Accuracy, Confusion Matrix, and Classification Report. Based on performance metrics, I selected the best-performing model for final deployment."

In [6]:
# ==============================
# IMPORT LIBRARIES
# ==============================
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

# ==============================
# DEFINE FEATURES & TARGET
# ==============================
X = data.drop("Cover_Type", axis=1)
y = data["Cover_Type"]

# ==============================
# TRAIN TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==============================
# INITIALIZE MODELS
# ==============================
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    
    # Pipeline added here (scaling required)
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    
    # Pipeline added here (scaling required)
    "KNN": Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),
    
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# ==============================
# TRAIN & EVALUATE
# ==============================
results = {}

for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred)
    
    results[name] = acc
    
    print("="*50)
    print(f"{name}")
    print("="*50)
    print("Accuracy:", acc)
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", cr)

# ==============================
# FINAL COMPARISON
# ==============================
print("\nModel Accuracy Comparison:")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

Random Forest
Accuracy: 0.9433134553430667

Confusion Matrix:
 [[  459     0     5     0   144     4     2]
 [    0   405    15     0     2    10     0]
 [    2    18   333     0    31    48     0]
 [    1     0     0   409     0     0    22]
 [   26     0    10     5 20293     5   275]
 [    3    21    59     0    34   315     0]
 [    9     0     2    42   859     0  5310]]

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.75      0.82       614
           1       0.91      0.94      0.92       432
           2       0.79      0.77      0.78       432
           3       0.90      0.95      0.92       432
           4       0.95      0.98      0.97     20614
           5       0.82      0.73      0.77       432
           6       0.95      0.85      0.90      6222

    accuracy                           0.94     29178
   macro avg       0.89      0.85      0.87     29178
weighted avg       0.94      0.94      0.94     29178


# Random Forest performed best with 94% accuracy and strong class-wise performance, so I selected it as the final model.
# Decision Tree performed well but slightly lower than Random Forest due to overfitting tendency.
# XGBoost gave competitive performance but slightly lower than Random Forest in my dataset.
# KNN performed moderately but was less accurate compared to tree-based models.
# Logistic Regression performed lowest because the dataset has complex non-linear patterns.

# Important Insight from Confusion Matrix

# Class 4 has very high support (20,614 samples) → model predicts it very well.
# Smaller classes like 0, 2, and 5 are slightly harder to predict.
# Random Forest handled class imbalance better than others.

# I built five classification models and compared them using Accuracy, Confusion Matrix, and Classification Report. Random Forest achieved the highest accuracy of 94.33% with strong precision and recall across classes. Therefore, I selected Random Forest as the final model for deployment.

In [11]:
# ==============================
# IMPORT LIBRARIES
# ==============================
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# ==============================
# DEFINE FEATURES & TARGET
# ==============================
X = data.drop("Cover_Type", axis=1)
y = data["Cover_Type"]

# ==============================
# DEFINE STRATIFIED K-FOLD
# ==============================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ==============================
# RANDOM FOREST
# ==============================
rf = RandomForestClassifier(random_state=42)
rf_scores = cross_val_score(rf, X, y, cv=skf, scoring='accuracy')

print("Random Forest CV Accuracy:", rf_scores.mean())

# ==============================
# DECISION TREE
# ==============================
dt = DecisionTreeClassifier(random_state=42)
dt_scores = cross_val_score(dt, X, y, cv=skf, scoring='accuracy')

print("Decision Tree CV Accuracy:", dt_scores.mean())

# ==============================
# LOGISTIC REGRESSION (with Pipeline)
# ==============================
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

lr_scores = cross_val_score(lr_pipeline, X, y, cv=skf, scoring='accuracy')

print("Logistic Regression CV Accuracy:", lr_scores.mean())

# ==============================
# KNN (with Pipeline)
# ==============================
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

knn_scores = cross_val_score(knn_pipeline, X, y, cv=skf, scoring='accuracy')

print("KNN CV Accuracy:", knn_scores.mean())

# ==============================
# XGBOOST
# ==============================
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_scores = cross_val_score(xgb, X, y, cv=skf, scoring='accuracy')

print("XGBoost CV Accuracy:", xgb_scores.mean())

Random Forest CV Accuracy: 0.943745287545411
Decision Tree CV Accuracy: 0.9281650558640072
Logistic Regression CV Accuracy: 0.8184591130303653
KNN CV Accuracy: 0.8858112276372611
XGBoost CV Accuracy: 0.9299403660292003


# Based on cross-validation results, Random Forest is selected as the best-performing model for Cover Type classification.

In [12]:
import pickle
best_model = models["Random Forest"]

with open('models.pkl', 'wb') as file:
    pickle.dump(best_model, file)

In [14]:
sample_data = pd.DataFrame([[2596.0, 510.0, 6279.0, 1.173173,
                             -0.727312, 0.258124, -0.915049,
                             -0.013352, -4560.885708, 49.468393,
                             150.0, 148.0, 0.014542,
                             12.0,
                             0, 0, 1, 0, 0, 0]],
                           columns=X.columns)

prediction = best_model.predict(sample_data)

print("Predicted Cover Type:", prediction)

Predicted Cover Type: [0]


# After selecting the best model, I added prediction code to classify new forest data into the correct Cover Type. This completes the end-to-end ML workflow from training to deployment.

# The model predicted class 0, which corresponds to Spruce/Fir forest type. Based on the environmental conditions provided, the area is most suitable for Spruce/Fir vegetation.

# 📌 Hyperparameter Tuning for RandomForestClassifier using RandomizedSearchCV

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 40}
Best CV Accuracy: 0.9329460552471041


# After identifying Random Forest as the best-performing model, I performed hyperparameter tuning using RandomizedSearchCV to optimize model accuracy and reduce overfitting. I then retrained the model using the best parameters on the full dataset and saved it as a .pkl file for deployment.

In [16]:
best_model = random_search.best_estimator_

best_model.fit(X, y)

,n_estimators,100
,criterion,'gini'
,max_depth,40
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Evaluate on Test Data

In [17]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 1.0

Confusion Matrix:
 [[  614     0     0     0     0     0     0]
 [    0   432     0     0     0     0     0]
 [    0     0   432     0     0     0     0]
 [    0     0     0   432     0     0     0]
 [    0     0     0     0 20614     0     0]
 [    0     0     0     0     0   432     0]
 [    0     0     0     0     0     0  6222]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       614
           1       1.00      1.00      1.00       432
           2       1.00      1.00      1.00       432
           3       1.00      1.00      1.00       432
           4       1.00      1.00      1.00     20614
           5       1.00      1.00      1.00       432
           6       1.00      1.00      1.00      6222

    accuracy                           1.00     29178
   macro avg       1.00      1.00      1.00     29178
weighted avg       1.00      1.00      1.00     29178



In [18]:
best_model.fit(X, y)

,n_estimators,100
,criterion,'gini'
,max_depth,40
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
from joblib import dump
dump(best_model, 'final_cover_type_model.joblib')

['final_cover_type_model.joblib']

# After hyperparameter tuning, I selected the best-performing model based on cross-validation and test accuracy. I then retrained it on the full dataset and saved it as a .pkl file using joblib for deployment purposes.

In [22]:
import os
os.listdir()

['.ipynb_checkpoints',
 'classification.csv',
 'classification.ipynb',
 'cleaned_data.csv',
 'cover_type (1).csv',
 'datacleaning.ipynb',
 'data_preprocessing.ipynb',
 'final_cover_type_model.joblib',
 'label_encoder.pkl',
 'models.pkl',
 'streamlit.py']